# B1 — Nested Discretization 구조 검증


        B1은 **격자를 촘촘하게 만들 때 이전 단계의 위치와 물리적 움직임이
        그대로 포함되는지** 확인한다. 쉽게 말해 L0에서 가능했던 움직임을
        L1/L2가 잃어버리지 않고, virtual switching edge가 실제 grid node에
        정확히 연결되는지 검사하는 단계다.

In [1]:
from pathlib import Path
import json
from html import escape
import subprocess
import sys
from IPython.display import display, Markdown, Image, HTML, FileLink

def locate_repo_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "p1b_4D").is_dir() and (candidate / "p1b_roadmap_0729.md").exists():
            return candidate
    raise RuntimeError("glider_hybrid_control repository root를 찾지 못했습니다.")

ROOT = locate_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RESULTS = ROOT / "results"
STAGE = "B1"
STATUS = "COMPLETED"

def run_module(module, *arguments, timeout=None):
    command = [sys.executable, "-m", module, *map(str, arguments)]
    completed = subprocess.run(
        command, cwd=ROOT, text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, timeout=timeout,
    )
    print(completed.stdout)
    if completed.returncode != 0:
        raise RuntimeError(f"{module} 실행 실패: exit={completed.returncode}")
    return completed.stdout

def load_json(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"결과 파일이 없습니다: {path}")
    return json.loads(path.read_text(encoding="utf-8"))

def show_png(path, width=1050):
    path = Path(path)
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        display(Markdown(f"> ⚠️ 그림이 없습니다: `{path}`"))

def display_table(rows, columns=None):
    if isinstance(rows, dict):
        rows = [rows]
    rows = list(rows)
    if columns is None:
        columns = []
        for row in rows:
            for key in row:
                if key not in columns:
                    columns.append(key)
    if not rows:
        display(Markdown("_(표시할 행이 없습니다.)_"))
        return
    def cell(value):
        if isinstance(value, float):
            value = f"{value:.8g}"
        elif isinstance(value, (dict, list, tuple)):
            value = json.dumps(value, ensure_ascii=False)
        return escape(str(value))
    header = "".join(f"<th>{cell(name)}</th>" for name in columns)
    body = "".join(
        "<tr>" + "".join(f"<td>{cell(row.get(name, ''))}</td>" for name in columns) + "</tr>"
        for row in rows
    )
    display(HTML(
        "<div style='overflow-x:auto'><table>"
        f"<thead><tr>{header}</tr></thead><tbody>{body}</tbody></table></div>"
    ))

display(Markdown(f"**{STAGE} 상태:** `{STATUS}`  \nRepository: `{ROOT}`"))

**B1 상태:** `COMPLETED`  
Repository: `C:\Users\jeffe\Desktop\git\glider_hybrid_control`

## 이 노트북에서 확인하는 것

- L0 ⊂ L1 ⊂ L2 spatial grid
- physical edge와 speed/action family의 nesting
- virtual switching target의 nesting
- endpoint snapping 없이 machine-precision endpoint 도달
- common high-fidelity evaluator qualification 계약

In [2]:
from p1b_4D.direction_b_discretization import (
    DIRECTION_B_GRID_COUNTS, DIRECTION_B_SPEEDS,
    DIRECTION_B_PRODUCTION_CONFIGURATION_ID,
)

rows = []
enriched_offsets = {0: 3, 1: 9, 2: 33}
for terrain, levels in DIRECTION_B_GRID_COUNTS.items():
    for level, (nz, nh) in enumerate(levels):
        rows.append({
            "terrain": terrain, "level": f"L{level}",
            "z nodes": nz, "h nodes": nh,
            "position nodes": nz * nh,
            "enriched directions": enriched_offsets[level],
            "V5 actions/state": enriched_offsets[level] * len(DIRECTION_B_SPEEDS["V5"]),
        })
display_table(rows)
print("B4 production configuration:", DIRECTION_B_PRODUCTION_CONFIGURATION_ID)

terrain,level,z nodes,h nodes,position nodes,enriched directions,V5 actions/state
single_hill,L0,161,101,16261,3,15
single_hill,L1,321,201,64521,9,45
single_hill,L2,641,401,257041,33,165
two_hill,L0,81,51,4131,3,15
two_hill,L1,161,101,16261,9,45
two_hill,L2,321,201,64521,33,165
goal_in_valley,L0,117,51,5967,3,15
goal_in_valley,L1,233,101,23533,9,45
goal_in_valley,L2,465,201,93465,33,165


B4 production configuration: direction_b_l2_enriched_v9_q9_e1025


## Regression 실행

In [3]:
RUN_TESTS = True
if RUN_TESTS:
    output = run_module(
        "unittest", "p1b_4D.test_direction_b_discretization"
    )
    display(Markdown("✅ **B1 entry-gate regression 통과**"))
else:
    display(Markdown("검사를 건너뛰었습니다. `RUN_TESTS=True`로 변경하세요."))

2026-07-30 09:44:45,598 | INFO | stackelberg | phase=Phase 1: Configuration status=started
2026-07-30 09:44:45,599 | WARNING | stackelberg | phase=Phase 1: Configuration warning=Vehicle segment_length is deferred to the future transcription contract instead of fabricating a Phase 1 value
2026-07-30 09:44:45,599 | INFO | stackelberg | phase=Phase 1: Configuration status=success elapsed_seconds=0.000380
.2026-07-30 09:44:45,610 | INFO | stackelberg | phase=Phase 1: Configuration status=started
2026-07-30 09:44:45,610 | WARNING | stackelberg | phase=Phase 1: Configuration warning=Vehicle segment_length is deferred to the future transcription contract instead of fabricating a Phase 1 value
2026-07-30 09:44:45,610 | INFO | stackelberg | phase=Phase 1: Configuration status=success elapsed_seconds=0.000243
2026-07-30 09:44:45,611 | INFO | stackelberg | phase=Phase 1: Configuration status=started
2026-07-30 09:44:45,611 | WARNING | stackelberg | phase=Phase 1: Configuration warning=Vehicle seg

✅ **B1 entry-gate regression 통과**

## 직관적 결론

L0→L1→L2는 서로 무관한 세 solver가 아니다. 동일한 물리 문제를
더 세밀하게 보는 nested family다. 따라서 뒤 단계의 차이를
grid/action resolution 변화로 해석할 수 있다.